# GeoLifeCLEF 2025 — one environmental challenger run
This master notebook runs only the current experiment. Historical stages remain in Git and the experiment log; Run All does not repeat them.

Question: can actual competition environmental predictors, preserved time-variable identity and reflectance-aware imagery improve on our previous fusion architecture? This is a challenger, not a verified SOTA model.

Competition PA data only, all observed PA species retained, no external data or pretrained weights. Shared spatial partitions and preprocessing. Italy and Switzerland remain untouched country audits. No refit after policy calibration.

Best previous official private score: 0.18900; version 19: 0.17516. Competition-winning private target: 0.2302. Internal scores do not establish that this target is beaten.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
os.environ['GLC_PIPELINE_STARTED_AT'] = str(time.time())
os.environ['PYTHONUNBUFFERED'] = '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').is_file(), 'Run from the repository; API deployment embeds tracked source automatically.'
DATA_ROOT = next((p for p in [Path('/kaggle/input/competitions/geolifeclef-2025'), Path('/kaggle/input/geolifeclef-2025')] if p.is_dir()), None)
assert DATA_ROOT is not None, 'Attach the geolifeclef-2025 competition.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pytest'], cwd=ROOT, check=True)


## Single bounded pipeline
1. Join EnvironmentalValues PA tables by surveyId, reject missing/duplicate IDs, fit normalization on the training partition only.
2. Train one fusion reference and two challenger seeds, up to 24 epochs each. Checkpoint selection is separate from policy calibration and final audit.
3. Freeze checkpoints; choose ensemble weight and output policy only on calibration surveys, including a zero-challenger fallback.
4. Report untouched spatial/country results and a one-seed architectural comparison. No audit-based retuning or post-calibration refit.
5. Generate ordered official test predictions and their SHA-256 manifest. This notebook does not automatically submit to the competition.

The 10.5-hour guard includes setup, data preparation, training and inference. Insufficient budget or missing mandatory data causes an explicit failure. Half-degree spatial blocks are not a distance buffer.

In [ ]:
remaining = max(1, 10.5 * 3600 - (time.time() - float(os.environ['GLC_PIPELINE_STARTED_AT'])))
subprocess.run([sys.executable, '-m', 'scripts.run_environmental_challenger', '--data-root', str(DATA_ROOT), '--epochs', '24', '--batch-size', '128', '--max-hours', '10.5'], cwd=ROOT, check=True, timeout=remaining)
report = json.loads((ROOT / 'artifacts/environmental_challenger/challenger_report.json').read_text())
{key: report[key] for key in ('selected', 'untouched_audit', 'submission', 'total_pipeline_hours', 'sota_proven')}


## Research provenance and limits
Public comparator: [2025 Model GeoLifeCLEF](https://www.kaggle.com/code/lonansyayf/2025-model-geolifeclef), linked by its [CLEF working note](https://ceur-ws.org/Vol-4038/paper_255.pdf). It is not the winner and uses ImageNet initialization, which is not imported here.

Top approaches informed the emphasis on environmental data and complementary modalities: [PredComX](https://ceur-ws.org/Vol-4038/paper_261.pdf), [Tighnari](https://arxiv.org/abs/2602.08282). Their full compute/data requirements are not reproduced.

This is an experimental adaptation, not a reproduction. PO representation pretraining is not included. An internal win is neither a paper contribution by itself nor proof of competition-leading performance.